# 08 · Distributed Troubles — Networks Lie, Clocks Lie, Processes Freeze

**Part 1 · Data-Intensive Applications** · *Prerequisites: 05* · *Book mapping: DDIA ch. 8*

On one machine, things either work or fail — a function returns or the process is dead.
Distribute the system and you enter the world of **partial failure**: some part is broken,
you cannot reliably tell *which* part, and — the truly cruel bit — you cannot even tell
*whether*. A request that gets no answer admits at least five explanations (request lost;
request queued; remote crashed; remote slow; reply lost), and they are **indistinguishable
from the sender's chair**.

This notebook is deliberately a parade of horrors, each built and measured, because the
defenses (nb 09's consensus, and every retry/idempotency pattern later) only make sense once
you've watched the attacks succeed.

**Learning objectives.**

1. Build a timeout-based **failure detector** and demonstrate its inherent false positives;
   quantify the timeout dilemma (fast detection ↔ false alarms).
2. Reproduce the **zombie leader**: a process pause turns a valid check into an invalid
   action; then defeat it with **fencing tokens**.
3. Reproduce silent data loss from **clock skew** under last-write-wins.
4. State the system models (async network, crash-recovery, non-Byzantine) that real designs
   assume, and what "truth by majority" means.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from sysdes import viz
from sysdes.sim import Network, Node, Sim

viz.use_style()
rng = np.random.default_rng(seed=8)

## 1. You cannot tell slow from dead

The only universal detection tool is the **timeout** — and a timeout is a *bet*, not a fact.
The five indistinguishable fates of a request:

In [ ]:
fig, ax = viz.message_timeline(
    actors=["client", "server"],
    messages=[
        {"src": "client", "dst": "server", "send": 0, "recv": 3, "label": "case A: request lost",
         "lost": True},
        {"src": "client", "dst": "server", "send": 5, "recv": 8, "label": "case B: server slow/paused",
         "color": "#ef6c00"},
        {"src": "client", "dst": "server", "send": 11, "recv": 14, "label": "case C"},
        {"src": "server", "dst": "client", "send": 15, "recv": 18, "label": "case C: reply lost",
         "lost": True, "color": "#6a1b9a"},
    ],
    events=[{"t": 9.5, "actor": "server", "label": " B: reply comes... eventually",
             "color": "#ef6c00"},
            {"t": 19, "actor": "client", "label": " all cases: silence + timeout",
             "color": "#c62828"}],
    title="Five failures, one observation: no reply",
)
plt.show()

Note the deep consequence hiding in case A vs C: on timeout you don't know whether the
operation **happened**. Retrying may execute it twice; not retrying may mean it never ran.
(That's why *idempotency* becomes a load-bearing word in nb 11.)

### 1.1 Build: a failure detector — and its false accusation

A node pings its peer every 10 time units and suspects it dead if no pong arrives within 15.
We run three acts: normal network → a latency spike (the peer is *fine*, the network is
congested) → a real crash.

In [ ]:
class Ponger(Node):
    def handle(self, src, msg):
        if msg["type"] == "ping":
            self.send(src, {"type": "pong", "n": msg["n"]})

class Pinger(Node):
    """Timeout-based failure detector. Suspicion = a timeout fired, nothing more."""

    def __init__(self, name, target, interval=10.0, timeout=15.0):
        super().__init__(name)
        self.target, self.interval, self.timeout = target, interval, timeout
        self.pending: dict[int, object] = {}   # ping n -> its timeout event
        self.suspicions: list[float] = []      # times at which we declared 'dead!'
        self.n = 0

    def start(self):
        self.n += 1
        self.send(self.target, {"type": "ping", "n": self.n})
        self.pending[self.n] = self.after(self.timeout, self._timeout, self.n)
        self.after(self.interval, self.start)                  # ping forever

    def _timeout(self, n):
        if n in self.pending:                                  # no pong in time
            del self.pending[n]
            self.suspicions.append(self.sim.now)

    def handle(self, src, msg):
        if msg["type"] == "pong":
            event = self.pending.pop(msg["n"], None)
            if event:
                event.cancel()                                 # alive after all

sim = Sim()
net = Network(sim, latency=(1.0, 4.0), seed=1)
detector, peer = Pinger("D", "P"), Ponger("P")
net.add(detector), net.add(peer)

sim.at(0.0, detector.start)
# Act 2 (t=100..140): congestion - round trips of 50-70, peer perfectly alive.
sim.at(100.0, lambda: setattr(net, "latency", (25.0, 35.0)))
sim.at(140.0, lambda: setattr(net, "latency", (1.0, 4.0)))
# Act 3 (t=200): the peer actually dies.
sim.at(200.0, net.partition, {"D"}, {"P"})
sim.run(until=300.0)

false_alarms = [t for t in detector.suspicions if t < 200]
true_alarms = [t for t in detector.suspicions if t >= 200]
print(f"suspicions during congestion (peer ALIVE): {[f'{t:.0f}' for t in false_alarms]}")
print(f"first suspicion after the real crash at t=200: t={true_alarms[0]:.0f}")

assert not [t for t in detector.suspicions if t < 100], "quiet network: no false alarms"
assert false_alarms, "congestion alone produced 'dead!' verdicts - FALSE POSITIVES"
assert true_alarms[0] <= 200 + 10 + 15, "real crash detected within interval+timeout"

The false alarms are not a bug in our detector — they are the *theorem*: in an asynchronous
network no detector can be both **accurate** (never wrongly suspect) and **fast**. Every
timeout value picks a point on that trade-off:

In [ ]:
# Round-trip times with a heavy tail (congestion, retransmits, GC on the peer...):
rtts = rng.lognormal(mean=1.0, sigma=0.8, size=300_000)

timeouts = np.linspace(2, 40, 60)
false_positive_rate = np.array([(rtts > t).mean() for t in timeouts])

fig, ax1 = plt.subplots()
ax1.plot(timeouts, false_positive_rate * 100, color="#c62828")
ax1.set_yscale("log")
ax1.set_xlabel("timeout")
ax1.set_ylabel("false alarms per 100 probes (log)", color="#c62828")
ax2 = ax1.twinx()
ax2.plot(timeouts, timeouts, color="#1565c0", linestyle="--")
ax2.set_ylabel("time to detect a real crash", color="#1565c0")
ax2.grid(False)
ax1.set_title("The timeout dilemma: pick your poison")
plt.show()

assert all(a >= b for a, b in zip(false_positive_rate, false_positive_rate[1:])), \
    "longer timeout -> fewer false alarms"
p999 = float(np.percentile(rtts, 99.9))
print(f"e.g. timeout at RTT p99.9 ({p999:.1f}) -> "
      f"{(rtts > p999).mean():.2%} false alarms, {p999:.1f} units detection delay")

Why false positives are *expensive*, not just embarrassing: "dead" verdicts trigger
failovers (nb 05) and rebalancing (nb 06) — heavy machinery that, launched at a healthy
node during congestion, adds load to a struggling system. Cascading failures are often
failure *detection* gone feral. Production practice: adaptive timeouts from measured RTT
distributions (jittered), and treating suspicion as a *degree*, not a verdict.

## 2. Process pauses and the zombie leader

Between any two lines of your program, the world may move on: garbage collection stops the
process for seconds; a VM migrates; a laptop lid closes; a page-in stalls. This murders
**check-then-act** logic:

```text
if lease.still_valid():     # true at this instant...
    <-- pause: 20 seconds of GC -->
    write_to_storage()      # ...executed in a world where it isn't
```

The node isn't malicious or crashed — it is a **zombie**: it *believes* it holds a lease
that has long expired, and its next action lands like a wrecking ball. Build it:

In [ ]:
class LockService(Node):
    """Grants time-limited leases. Fencing: every grant carries an increasing token."""

    def __init__(self, name, ttl=30.0):
        super().__init__(name)
        self.ttl = ttl
        self.expires = -1.0
        self.token = 0

    def handle(self, src, msg):
        if msg["type"] == "acquire":
            if self.sim.now >= self.expires:               # lease free (or expired)
                self.token += 1
                self.expires = self.sim.now + self.ttl
                self.send(src, {"type": "granted", "token": self.token,
                                "expires": self.expires})
            else:
                self.send(src, {"type": "denied"})

class Storage(Node):
    """The resource the lease protects. `fencing=True` -> reject stale tokens."""

    def __init__(self, name, fencing: bool):
        super().__init__(name)
        self.fencing = fencing
        self.value = None
        self.highest_token = 0
        self.rejected: list[tuple] = []

    def handle(self, src, msg):
        if msg["type"] == "write":
            if self.fencing and msg["token"] < self.highest_token:
                self.rejected.append((src, msg["token"]))  # zombie, begone
                return
            self.highest_token = max(self.highest_token, msg["token"])
            self.value = (msg["value"], src)

class Worker(Node):
    """Acquires the lease, then writes - after an optional GC pause that sits
    exactly in the check-then-act gap."""

    def __init__(self, name, pause=0.0):
        super().__init__(name)
        self.pause = pause
        self.lease = None

    def acquire(self):
        self.send("lock", {"type": "acquire"})

    def handle(self, src, msg):
        if msg["type"] == "granted":
            self.lease = msg
            # THE CHECK: lease valid? (It is - it was granted a moment ago.)
            assert self.sim.now < msg["expires"]
            # THE PAUSE: everything freezes; the scheduled send happens later.
            self.after(self.pause, self.send, "store",
                       {"type": "write", "value": f"data-from-{self.name}",
                        "token": msg["token"]})

def zombie_scenario(fencing: bool):
    sim = Sim()
    net = Network(sim, latency=(2.0, 2.0), seed=0)
    lock, store = LockService("lock", ttl=30.0), Storage("store", fencing)
    a, b = Worker("A", pause=60.0), Worker("B", pause=0.0)   # A gets a 60-unit GC pause
    for n in (lock, store, a, b):
        net.add(n)
    sim.at(0.0, a.acquire)      # A holds lease [~2, ~32], token 1
    sim.at(40.0, b.acquire)     # lease expired -> B gets token 2, writes promptly
    sim.run()
    return store

store = zombie_scenario(fencing=False)
print(f"without fencing, final value: {store.value}")
assert store.value == ("data-from-A", "A"), \
    "the ZOMBIE (lease expired 30+ units ago) overwrote the rightful holder's write"

store = zombie_scenario(fencing=True)
print(f"with fencing, final value: {store.value}, rejected: {store.rejected}")
assert store.value == ("data-from-B", "B"), "the current holder's write stands"
assert store.rejected and store.rejected[0][1] == 1, "the stale token was refused"

The moral generalizes far beyond locks: **a node's belief about its own role is never
sufficient**. "I am the leader / I hold the lock / my lease is valid" can all be stale the
instant after they're checked. Safety must be enforced *at the resource*, by something the
zombie cannot fake — a monotonically increasing token issued with the grant. (Notice the
storage does the enforcing; a lock service alone, however correct, cannot save you.)

## 3. Clocks — two kinds, both treacherous

- **Time-of-day (wall) clocks**: synchronized via NTP, which *slews and steps* them —
  including **backwards**. Typical sync error: milliseconds on a good LAN, tens to hundreds
  of ms in the wild. Never measure durations with them; never order events across machines
  with them.
- **Monotonic clocks**: guaranteed to move forward, great for durations/timeouts — but their
  absolute value is meaningless and **incomparable across machines**.

In [ ]:
# Measuring a duration across an NTP step (wall clock stepped back 3s mid-measurement):
wall_start = 1000.000
wall_end = 1002.5 - 3.0            # 2.5s of real time passed, then NTP stepped -3s
elapsed_wall = wall_end - wall_start
mono_start, mono_end = 555.000, 557.500          # monotonic: unaffected
print(f"wall-clock 'duration': {elapsed_wall:+.1f}s   monotonic duration: "
      f"{mono_end - mono_start:.1f}s")
assert elapsed_wall < 0, "wall clocks can measure NEGATIVE durations"
assert mono_end - mono_start > 0

### Failure first: clock skew + last-write-wins = silent loss

Notebook 05 warned that LWW resolves conflicts by trusting timestamps. Here is the bill.
Node A's clock runs 50 units fast. Node B writes *after* A in real time — even *after
reading A's value* (a true causal successor) — but B's honest clock stamps it lower:

In [ ]:
class LWWStore:
    """Keep whichever write carries the highest timestamp. What could go wrong."""
    def __init__(self):
        self.data: dict[str, tuple[float, str]] = {}
        self.discarded: list[tuple[float, str]] = []

    def put(self, key, value, ts):
        if key not in self.data or ts > self.data[key][0]:
            if key in self.data:
                self.discarded.append(self.data[key])
            self.data[key] = (ts, value)
        else:
            self.discarded.append((ts, value))       # silently dropped

CLOCK_SKEW_A = +50.0     # A's wall clock runs fast; B's is correct

store = LWWStore()
# Real time 10: A sets the config...
store.put("config", "mode=fast", ts=10.0 + CLOCK_SKEW_A)     # stamped 60
# Real time 20: B READS A's value, decides it's wrong, and overwrites it:
assert store.data["config"][1] == "mode=fast"
store.put("config", "mode=safe", ts=20.0)                    # stamped 20

winner_ts, winner = store.data["config"]
print(f"final config: {winner!r} (ts={winner_ts}); discarded: {store.discarded}")
assert winner == "mode=fast", \
    "B's LATER, CAUSALLY-INFORMED write was silently discarded - 50 units of skew ate it"

No error was raised anywhere; the fix that "fixed" the config simply doesn't exist anymore.
Defenses, in increasing strength: bounded-skew infrastructure with **uncertainty intervals**
(Google TrueTime exposes [earliest, latest] and *waits out* the uncertainty before
committing); **logical clocks** that track causality instead of pretending to know the time
(Lamport/vector clocks — built in nb 09); or not using LWW for anything you can't afford to
lose (nb 05's siblings).

## 4. What, then, can be trusted?

The models real systems assume (and this course assumes from here on): **asynchronous
network** (no bound on delay), **crash-recovery nodes** (they die and come back with stable
storage intact, without warning), **non-Byzantine** (nodes are honest-but-faulty; corruption
is caught by checksums, identity by auth — but no node *lies strategically*). Byzantine
fault tolerance — surviving nodes that lie — costs 3f+1 replicas and heavyweight protocols;
outside blockchains and aerospace, the standard bet is that your own fleet is buggy, not
treasonous.

Within that model, one principle survives everything this notebook threw at us: **truth is
decided by majority**. A node cannot know *about itself* whether it is alive-and-connected
or zombified — but a **quorum** (nb 05) of independent nodes, each fallible, can jointly
decide "node X is dead", "lease belongs to Y, at token 7", "this write committed" — and
provided fewer than half are faulty, the decision is safe *and* nobody's pause or skewed
clock can overturn it alone. Turning that principle into working machinery — locks, leader
election, atomic commit — is **consensus**, and it's next.

## 5. Exercises

1. **Timeout budgeting.** Your RTTs are lognormal with median 3 ms and σ=0.8. You require
   < 0.05% false suspicions per probe. Estimate the needed timeout from the distribution
   (use the sampled `rtts`), and the resulting worst-case detection time. What operational
   fact could invalidate your number a month from now?
2. **Monotonic across machines.** Node A's monotonic clock reads 10^9; node B's reads 42.
   A message from A carries `mono=10^9`. Why is comparing it with B's clock meaningless,
   and what *is* the legitimate cross-machine use of monotonic time?
3. **TTL lock autopsy.** A Redis-style lock: `SET key token NX PX 30000`, holder deletes on
   completion. Walk the exact pause-based interleaving that corrupts the protected resource
   even though every component behaved to spec. Which two fixes work, and why is "make the
   TTL longer" not one of them?
4. **Skew accounting.** Two datacenters, NTP error up to ±100 ms each. Under LWW, what's
   the window in which causally-ordered cross-DC writes can be reversed? At 200 writes/s to
   shared keys, ballpark how many at-risk writes per day. What ordering primitive removes
   the risk entirely?
5. **Byzantine or not?** Classify + name the defense: (a) a disk returns bit-flipped data;
   (b) an NTP server feeds a wrong time; (c) a compromised node reports fake replication
   progress; (d) a buggy client library sends malformed requests.

### Solutions

**1.** Timeout ≈ the RTT's 99.95th percentile — from our sample ≈ `np.percentile(rtts,
99.95)` ≈ 30–40 units (ms-scale in the story), giving detection delay of the same size.
Invalidator: the RTT distribution isn't stationary — new congestion patterns, a new hop, a
chattier neighbor — which is why real detectors *re-estimate* percentiles continuously
(φ-accrual style) instead of hard-coding last month's tail.

**2.** Monotonic clocks count from an arbitrary per-boot origin; only *differences on the
same machine* mean anything. Cross-machine, the legitimate pattern is: A computes a
*duration* or deadline locally ("valid for 10 s") and ships the duration, never the reading;
B applies it to its own clock — accepting that transit time blurs it.

**3.** Client A acquires (token T, TTL 30 s) → checks success → GC pause 40 s → lock
expires; client B acquires (token T') and starts writing → A wakes and writes: corruption.
Everyone was in-spec: the pause is invisible to A. Fixes: **fencing tokens** enforced by the
resource (monotonic acquire counter; resource rejects < highest seen), or **CAS on the
resource itself** so the write validates ownership atomically at the resource. Longer TTL
merely moves the race (any finite TTL loses to a longer pause) and worsens recovery from
real crashes.

**4.** Combined uncertainty up to ~200 ms: any two causally-ordered writes closer than that
can be timestamp-reversed. At 200 writes/s, every write has a ~0.2 s danger window —
order-of-thousands of at-risk pairs per day on contended keys (exact count depends on
contention, but "not rare"). Removal: causality tracking (version vectors, nb 09) or
funnel writes for a key through one leader (nb 05/06) so order is assigned, not guessed.

**5.** (a) Not Byzantine in the protocol sense — fault, caught by checksums/CRCs
(end-to-end!). (b) Faulty *input* — bound it by querying multiple NTP sources and rejecting
outliers (which is a mini-majority-truth). (c) **Byzantine** — a node lying about protocol
state; ordinary consensus does not defend this; needs BFT or (realistically) attestation +
ops response. (d) Not Byzantine — untrusted input at the boundary; validate and fuzz
(protocol-aware input checking, nb 04's schemas help).

## 6. Takeaways

- Partial failure is *the* defining property; "no reply" carries five meanings and a timeout
  is a bet. Detection can be fast or accurate, never both — false "dead!" verdicts trigger
  heavy machinery, so treat suspicion gradually.
- **Check-then-act is broken** in a pausing world: any node can be a zombie. Enforce safety
  *at the resource* with fencing tokens — beliefs expire, monotonic tokens don't.
- Wall clocks step (even backwards) and disagree across machines; monotonic clocks are for
  local durations only. Timestamp ordering (LWW) silently discards causally-newer writes
  under skew.
- Assume async network, crash-recovery, non-Byzantine — and derive truth from **majorities**,
  never from a node's self-belief. That road leads directly to consensus.

**Further reading.** DDIA ch. 8; Kleppmann, *How to do distributed locking* (the Redlock
debate — fencing in the wild); Chandra & Toueg, *Unreliable Failure Detectors* (1996);
Google's Spanner paper §TrueTime.

**Next:** [09 · Consistency & consensus](09-consistency-consensus.ipynb) — the constructive
counterpart: what *can* be guaranteed, and the machinery (logical clocks, linearizability,
Raft) that guarantees it.